# S7 — Explore the PushT demonstration dataset
Inspect feature keys, frame dimensions, action dimensions, dataset FPS, and the expert action trajectory for the first episode. Run with the repository LeRobot Python environment.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lerobot.datasets.lerobot_dataset import LeRobotDataset

ds = LeRobotDataset('lerobot/pusht')
print('Episodes:', ds.num_episodes)
print('Frames:', ds.num_frames)
print('FPS:', ds.fps)
print('Features:')
for key, descriptor in ds.features.items():
    print(' -', key, descriptor)

In [ ]:
sample = ds[0]
print('Sample keys and shapes:')
for key, value in sample.items():
    print(f'  {key}: {tuple(value.shape) if hasattr(value, "shape") else type(value).__name__}')
print('Action dimensions:', sample['action'].numel())

In [ ]:
image_keys = [key for key in sample if 'image' in key and hasattr(sample[key], 'shape')]
print('Image keys:', image_keys)
if image_keys:
    img = sample[image_keys[0]].detach().cpu().numpy()
    if img.ndim == 3 and img.shape[0] in (1, 3, 4):
        img = np.transpose(img, (1, 2, 0))
    plt.figure(figsize=(5, 5))
    plt.imshow(np.clip(img, 0, 1) if img.dtype.kind == 'f' else img)
    plt.title(image_keys[0])
    plt.axis('off')
    plt.show()

In [ ]:
episode_id = int(sample['episode_index'].item())
actions, times = [], []
for i in range(ds.num_frames):
    row = ds[i]
    if int(row['episode_index'].item()) != episode_id:
        break
    actions.append(row['action'].detach().cpu().numpy())
    times.append(float(row['timestamp'].item()) if 'timestamp' in row else len(times) / ds.fps)
actions = np.stack(actions)
times = np.asarray(times) - times[0]
print(f'Episode {episode_id}: {len(actions)} frames; action shape {actions.shape}')
fig, ax = plt.subplots(figsize=(9, 4))
for dim in range(actions.shape[1]):
    ax.plot(times, actions[:, dim], label=f'action[{dim}]')
ax.set(xlabel='Time (s)', ylabel='Action value', title=f'PushT episode {episode_id}: expert actions')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## Record observations after execution
- Actual observation keys and shapes: **TODO**
- Image dimensions and interpretation: **TODO**
- Action dimensions and physical meaning: **TODO (do not infer units from dimensions)**
- Dataset FPS: **TODO**
- Notable variation in action traces: **TODO**
- How autonomous rollouts could encounter unfamiliar observations: **TODO**